# 03 – K-Means baseline (M2)

Phụ trách: M2 (Chu Thành Tân). Theo checklist M2, Phần 3:
- **3.1 Elbow**: chạy K = 2..15 trên từng lát cắt thời gian.
- **3.2 Silhouette, DBI, CH theo K** và đề xuất K.
- **3.3 Fit với K đã chốt (K = 5)**, ghi log qua `run_and_log`, lập bảng cụm, vẽ bản đồ Folium.

Dữ liệu đang dùng: **mẫu pilot tạm của M2** (`pilot_m2_v0`, 1 tuần). Khi M1 phát hành dữ liệu chính thức thì đổi `DATA_KEY = "model_input"` rồi chạy lại.

Lưu ý: toạ độ là centroid nên rất nhiều chuyến trùng toạ độ. Vì vậy chạy hai cách để đối chiếu:
- `full`: fit trên từng chuyến.
- `weighted`: fit trên các toạ độ duy nhất, `sample_weight` = số chuyến. Cùng hàm mục tiêu nhưng nhanh hơn.

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Du lieu that tren Colab: bo comment 2 dong duoi khi nhom da chot thu muc Drive
# from src.colab import link_drive_data
# link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## Tham số

In [ ]:
import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config, resolve_path
from src.models.kmeans import aggregate_unique, elbow, knee_point

cfg = load_config()
DATA_KEY = "pilot_m2"            # doi thanh "model_input" khi co du lieu cua M1
MANIFEST_KEY = "pilot_m2_manifest" if DATA_KEY == "pilot_m2" else "manifest"
SLICES = list(cfg["time_slices"])          # WEEKDAY_AM, WEEKDAY_PM, WEEKEND
KM = cfg["kmeans"]; SEED = cfg["project"]["seed"]
N_INIT_HIGH = 50                 # weighted chay re nen thu them n_init cao de tranh cuc tieu cuc bo

df = pd.read_parquet(resolve_path(cfg, DATA_KEY))
man = json.loads(resolve_path(cfg, MANIFEST_KEY).read_text(encoding="utf-8"))
SAMPLE_ID = man["sample_id"]
print("Du lieu:", SAMPLE_ID, "|", f"{len(df):,} diem", "| K =", KM["k_min"], "..", KM["k_max"], "| n_init =", KM["n_init"])

## 3.1 Elbow theo lát cắt

In [ ]:
rows = []
for s in SLICES:
    X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
    Xu, w, inv = aggregate_unique(X)
    common = dict(k_min=KM["k_min"], k_max=KM["k_max"], init=KM["init"], max_iter=KM["max_iter"], seed=SEED)
    runs = [
        ("full", KM["n_init"], elbow(X, n_init=KM["n_init"], **common)),
        ("weighted", KM["n_init"], elbow(Xu, n_init=KM["n_init"], sample_weight=w, **common)),
        ("weighted", N_INIT_HIGH, elbow(Xu, n_init=N_INIT_HIGH, sample_weight=w, **common)),
    ]
    for mode, n_init, e in runs:
        e = e.assign(time_slice=s, mode=mode, n_init=n_init, n_points=len(X), n_unique=len(Xu), sample_id=SAMPLE_ID)
        rows.append(e)
    print(f"{s:11s} {len(X):>7,} diem, {len(Xu):>4} toa do duy nhat | runtime full {runs[0][2].runtime_sec.sum():6.2f}s"
          f", weighted {runs[1][2].runtime_sec.sum():5.2f}s")

res = pd.concat(rows, ignore_index=True)[["sample_id", "time_slice", "mode", "n_init", "n_points", "n_unique",
                                          "k", "inertia", "n_iter", "converged", "runtime_sec"]]
print("Tat ca hoi tu:", bool(res.converged.all()))

### So sánh `full` và `weighted`

Cùng hàm mục tiêu, nên inertia tốt nhất phải gần bằng nhau. Chênh lệch dương nghĩa là `weighted` kém hơn `full` ở K đó (do rơi vào cực tiểu cục bộ khác).

In [ ]:
piv = res.pivot_table(index=["time_slice", "k"], columns=["mode", "n_init"], values="inertia")
piv.columns = [f"{m}_n{n}" for m, n in piv.columns]
best = piv.min(axis=1)
cmp = (piv.sub(best, axis=0).div(best, axis=0) * 100).round(3)
cmp.columns = [c + "_lech_%" for c in cmp.columns]
print("Lech so voi inertia tot nhat (%), lon nhat theo tung cach:")
print(cmp.max().to_string())
cmp.loc[(cmp > 0.01).any(axis=1)]

### Đường Elbow và điểm khuỷu gợi ý

Dùng inertia tốt nhất của mỗi K. Điểm khuỷu chỉ là **gợi ý** (khoảng cách lớn nhất tới dây cung); K cuối cùng chọn ở mục 3.2, có xét thêm Silhouette và bản đồ.

In [ ]:
knees = {}
fig, axes = plt.subplots(1, len(SLICES), figsize=(5 * len(SLICES), 4), sharey=False)
for ax, s in zip(axes, SLICES):
    b = best.loc[s]
    knees[s] = knee_point(b.index, b.values)
    ax.plot(b.index, b.values / 1e6, "o-", color="#1f77b4", label="inertia tot nhat")
    ax.axvline(knees[s], color="#d62728", ls="--", lw=1, label=f"khuyu goi y K={knees[s]}")
    n = int(res.loc[res.time_slice == s, "n_points"].iloc[0]); u = int(res.loc[res.time_slice == s, "n_unique"].iloc[0])
    ax.set_title(f"{s}  (n={n:,}, {u} toa do)")
    ax.set_xlabel("So cum K"); ax.set_ylabel("Inertia (km^2)")
    ax.set_xticks(b.index); ax.grid(alpha=.3); ax.legend()
fig.suptitle(f"K-Means Elbow - {SAMPLE_ID} (EPSG:26916, seed {SEED})")
fig.tight_layout()

fig_path = resolve_path(cfg, "figures_dir") / f"kmeans_elbow_{SAMPLE_ID}.png"
csv_path = resolve_path(cfg, "run_log").parent / f"kmeans_elbow_{SAMPLE_ID}.csv"
fig_path.parent.mkdir(parents=True, exist_ok=True); csv_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=150)
res.to_csv(csv_path, index=False)
print("Khuyu goi y:", knees)
print("Da luu:", csv_path.relative_to(cfg["_root"]), "|", fig_path.relative_to(cfg["_root"]))

### Mức giảm inertia khi tăng K

Tỷ lệ giảm so với K-1. Khi tỷ lệ này nhỏ dần, tăng thêm cụm không còn cải thiện nhiều.

In [ ]:
drop = (-best.groupby(level=0).pct_change() * 100).round(1).unstack(0)
drop

## 3.2 Silhouette, DBI, CH theo K

Mỗi K fit theo cách `weighted` với n_init=50 (đề xuất ở 3.1, chưa sửa config), rồi gán nhãn lại cho từng chuyến và tính chỉ số bằng `src.evaluation.evaluate`:
- `silhouette`: tính trên từng chuyến, mẫu con tối đa 20.000 chuyến, seed 42. Càng cao càng tốt.
- `silhouette_unique`: tính trên các toạ độ duy nhất, không trọng số. Dùng để đối chiếu, vì nhiều chuyến trùng toạ độ kéo Silhouette lên.
- `davies_bouldin`: càng thấp càng tốt. `calinski_harabasz`: càng cao càng tốt.
- `smallest_share`: tỷ trọng của cụm nhỏ nhất. Quá nhỏ nghĩa là có cụm vụn.

In [ ]:
from src.models.kmeans import scan_k, fit_weighted
K_RANGE = range(KM["k_min"], KM["k_max"] + 1)
scans = []
for s in SLICES:
    X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
    sc = scan_k(X, K_RANGE, init=KM["init"], n_init=N_INIT_HIGH, max_iter=KM["max_iter"], seed=SEED,
                silhouette_max_sample=cfg["evaluation"]["silhouette_max_sample"])
    scans.append(sc.assign(time_slice=s, sample_id=SAMPLE_ID, n_init=N_INIT_HIGH))
    print(f"{s:11s} xong ({sc.fit_sec.sum():.1f}s fit)")
scan = pd.concat(scans, ignore_index=True)
scan_path = resolve_path(cfg, "run_log").parent / f"kmeans_kscan_{SAMPLE_ID}.csv"
scan.to_csv(scan_path, index=False)
print("Da luu:", scan_path.relative_to(cfg["_root"]))

In [ ]:
metrics = [("silhouette", "Silhouette (tung chuyen) - cao tot"), ("silhouette_unique", "Silhouette (toa do duy nhat) - cao tot"),
           ("davies_bouldin", "Davies-Bouldin - thap tot"), ("calinski_harabasz", "Calinski-Harabasz - cao tot"),
           ("smallest_share", "Ty trong cum nho nhat")]
fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.6))
for ax, (m, title) in zip(axes, metrics):
    for s in SLICES:
        d = scan[scan.time_slice == s]
        ax.plot(d.k, d[m], "o-", ms=3, label=s)
    ax.axvspan(4.5, 6.5, color="grey", alpha=.12)
    ax.set_title(title, fontsize=9); ax.set_xlabel("K"); ax.grid(alpha=.3); ax.set_xticks(list(K_RANGE)[::2])
axes[0].legend(fontsize=8)
fig.suptitle(f"K-Means: chi so theo K - {SAMPLE_ID} (vung xam = khuyu Elbow K 5-6)")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"kmeans_kscan_{SAMPLE_ID}.png", dpi=150)

### Bảng ứng viên K = 4..8

In [ ]:
cand = scan[scan.k.between(4, 8)].set_index(["time_slice", "k"])[
    ["silhouette", "silhouette_unique", "davies_bouldin", "calinski_harabasz", "largest_share", "smallest_share"]]
print("K tot nhat theo tung chi so (K = 2..15):")
for s in SLICES:
    d = scan[scan.time_slice == s].set_index("k")
    print(f"  {s:11s} Silhouette max K={d.silhouette.idxmax()}, Sil.unique max K={d.silhouette_unique.idxmax()}, "
          f"DBI min K={d.davies_bouldin.idxmin()}, CH max K={d.calinski_harabasz.idxmax()}")
cand.round(3)

### Bản đồ tĩnh các ứng viên K = 5 và K = 6

Mỗi chấm là một toạ độ centroid, kích thước chấm tỷ lệ với số chuyến, màu là cụm, dấu X là tâm cụm. Bản đồ tương tác (Folium) làm ở mục 3.3.

In [ ]:
fig, axes = plt.subplots(2, len(SLICES), figsize=(5 * len(SLICES), 9.5))
for j, s in enumerate(SLICES):
    X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
    Xu, w, inv = aggregate_unique(X)
    for i, k in enumerate((5, 6)):
        lab, km = fit_weighted(X, k, init=KM["init"], n_init=N_INIT_HIGH, max_iter=KM["max_iter"], seed=SEED)
        ax = axes[i, j]
        ax.scatter(Xu[:, 0] / 1000, Xu[:, 1] / 1000, s=5 + 600 * w / w.max(), c=km.labels_, cmap="tab10",
                   vmin=0, vmax=9, alpha=.75, edgecolors="k", linewidths=.2)
        ax.scatter(km.cluster_centers_[:, 0] / 1000, km.cluster_centers_[:, 1] / 1000, marker="X", s=120, c="k")
        share = np.bincount(lab, minlength=k) / len(lab) * 100
        ax.set_title(f"{s} - K={k} | ty trong cum (%): " + ", ".join(f"{v:.0f}" for v in sorted(share, reverse=True)), fontsize=9)
        ax.set_aspect("equal"); ax.grid(alpha=.3); ax.set_xlabel("x (km, EPSG:26916)"); ax.set_ylabel("y (km)")
fig.suptitle(f"K-Means ung vien K=5 / K=6 - {SAMPLE_ID}")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"kmeans_candidates_{SAMPLE_ID}.png", dpi=130)

## 3.3 Fit K-Means với K đã chốt và bản đồ

- K lấy từ `config: kmeans.k_final` (chốt K = 5 ở mục 3.2). Dùng cách `weighted` với n_init=50.
- Mỗi lát cắt chạy qua `run_and_log`: đo runtime và RAM, tính chỉ số, lưu nhãn `outputs/labels/<run_id>.parquet`, ghi một dòng vào `outputs/metrics/run_log.csv`.
- Cụm được đánh số lại theo kích thước (cụm 0 là cụm lớn nhất) cho dễ đọc. Nhãn giữa các lát cắt không tương ứng với nhau, nên muốn ghép cụm giữa các lát cắt phải dựa vào toạ độ tâm (mục 5.1).
- **Mỗi lần chạy ô dưới sẽ ghi thêm 3 dòng vào run_log.csv**, đây là cách ghi có chủ đích (chỉ thêm, không sửa dòng cũ).

In [ ]:
from src.evaluation import run_and_log
from src.evaluation.hotspots import relabel_by_size, cluster_table
K_FINAL = KM["k_final"]
params = {"k": K_FINAL, "init": KM["init"], "n_init": N_INIT_HIGH, "max_iter": KM["max_iter"],
          "random_state": SEED, "mode": "weighted"}

def fit_fn(X):
    lab, km = fit_weighted(X, K_FINAL, init=KM["init"], n_init=N_INIT_HIGH, max_iter=KM["max_iter"], seed=SEED)
    assert km.n_iter_ < KM["max_iter"], "K-Means chua hoi tu"
    return relabel_by_size(lab)

final = {}
for s in SLICES:
    part = df[df.time_slice == s].reset_index(drop=True)
    X = part[["x_m", "y_m"]].to_numpy("float64")
    row = run_and_log(cfg, "kmeans", fit_fn, X, part.trip_id.to_numpy(), member="M2", time_slice=s,
                      sample_id=SAMPLE_ID, params=params, note="mau pilot tam M2; K chot o muc 3.2")
    lab = pd.read_parquet(resolve_path(cfg, "labels_dir") / f"{row['run_id']}.parquet")
    assert (lab.trip_id.values == part.trip_id.values).all()
    final[s] = (part, lab.label.to_numpy(), row)
    print(f"{row['run_id']}: {row['n_clusters']} cum, Silhouette {row['silhouette']:.3f}, DBI {row['davies_bouldin']:.3f}, "
          f"fit {row['runtime_fit_sec']:.3f}s, RAM them {row['mem_delta_mb']} MB")

### Bảng cụm

Tâm cụm là trung bình toạ độ theo từng chuyến (tức là tâm có trọng số). `radius_rms_m` là độ phân tán quanh tâm. `top_community_area` là khu vực (community area) xuất hiện nhiều nhất trong cụm.

In [ ]:
tabs = []
for s, (part, lab, row) in final.items():
    ct = cluster_table(part, lab, crs=cfg["project"]["crs"])
    tabs.append(ct.assign(time_slice=s, run_id=row["run_id"], sample_id=SAMPLE_ID))
clusters = pd.concat(tabs, ignore_index=True)
cl_path = resolve_path(cfg, "run_log").parent / f"kmeans_clusters_{SAMPLE_ID}.csv"
clusters.to_csv(cl_path, index=False)
print("Da luu:", cl_path.relative_to(cfg["_root"]))
show = clusters.assign(share_pct=(clusters.share * 100).round(1), radius_km=(clusters.radius_rms_m / 1000).round(2),
                       center=clusters.center_lat.round(4).astype(str) + ", " + clusters.center_lon.round(4).astype(str))
show[["time_slice", "cluster", "n_trips", "share_pct", "n_coords", "radius_km", "top_community_area", "center"]]

### Bản đồ tương tác

Mỗi lát cắt là một lớp, bật/tắt ở góc phải. File HTML cần Internet để tải nền bản đồ OpenStreetMap; khi thuyết trình thì dùng PNG dự phòng ở mục 3.2.

In [ ]:
from src.visualization.maps import cluster_map
map_path = resolve_path(cfg, "maps_dir") / f"kmeans_map_{SAMPLE_ID}.html"
layers = {f"{s} (K={K_FINAL})": (part, lab, clusters[clusters.time_slice == s]) for s, (part, lab, row) in final.items()}
cluster_map(layers, map_path, title=f"K-Means K={K_FINAL} - {SAMPLE_ID}",
            note="Mau pilot tam 1 tuan (17-23/04/2023), khong phai ket qua cuoi.")
print("Da luu:", map_path.relative_to(cfg["_root"]), f"({map_path.stat().st_size / 1024:.0f} KB)")

### Các dòng vừa ghi vào run_log.csv

In [ ]:
from src.evaluation import read_run_log
log = read_run_log(resolve_path(cfg, "run_log"))
ids = [row["run_id"] for _, _, row in final.values()]
log[log.run_id.isin(ids)][["run_id", "n_points", "n_clusters", "noise_ratio", "silhouette", "davies_bouldin",
                           "calinski_harabasz", "largest_share", "runtime_fit_sec", "mem_delta_mb", "machine"]]